# 🎬 IMDb Movie Reviews Sentiment Analysis
### End-to-End NLP Pipeline using TF-IDF & Logistic Regression
---
**Course / Project Title:** Sentiment Analysis on Movie Reviews  
**Objective:** Build a supervised Machine Learning model to automatically classify text reviews into **Positive** or **Negative** sentiments.

#### **Pipeline Workflow:**
1. **Data Acquisition & Exploration**
2. **Text Preprocessing & Cleaning**
3. **Feature Extraction (TF-IDF Vectorization)**
4. **Model Training (Logistic Regression & Naive Bayes)**
5. **Performance Evaluation (Confusion Matrix, Precision, Recall, F1-Score)**
6. **Live Inference Testing**

## 1. Import Dependencies & Load Dataset

In [ ]:
import re
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib

print("Libraries imported successfully!")

In [ ]:
# Load sample dataset or download full 50k IMDb dataset
dataset_path = 'data/imdb_50k.csv' if os.path.exists('data/imdb_50k.csv') else 'data/sample_reviews.csv'

df = pd.read_csv(dataset_path)
print(f"Loaded dataset from {dataset_path} with shape: {df.shape}")
df.head()

In [ ]:
# Check class distribution
plt.figure(figsize=(5, 3))
sns.countplot(x='sentiment', data=df, palette='viridis')
plt.title("Sentiment Class Distribution")
plt.show()

## 2. Text Preprocessing
Text data contains unwanted noise such as HTML markup, URLs, numbers, and punctuation. We clean each text entry before feature extraction.

In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = re.sub(r'<.*?>', ' ', text)                 # Remove HTML tags
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)  # Remove URLs
    text = text.lower()                                # Convert to lower case
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)            # Keep only letters
    words = text.split()                               # Normalize whitespace
    return " ".join(words)

# Apply cleaning
df['cleaned_review'] = df['review'].apply(clean_text)
df['label'] = df['sentiment'].apply(lambda x: 1 if str(x).lower().strip() == 'positive' else 0)

# Show sample before & after
print("Raw Review:", df['review'].iloc[0][:150])
print("Cleaned Review:", df['cleaned_review'].iloc[0][:150])

## 3. Dataset Splitting & TF-IDF Feature Extraction

In [ ]:
X = df['cleaned_review']
y = df['label']

# Stratified 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {len(X_train)} reviews")
print(f"Testing set:  {len(X_test)} reviews")

# TF-IDF Vectorizer with unigrams and bigrams
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"TF-IDF Matrix Shape: {X_train_tfidf.shape}")

## 4. Model Training & Comparison
We train both **Logistic Regression** and **Multinomial Naive Bayes** to compare their performance.

In [ ]:
# 1. Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_tfidf, y_train)
lr_pred = lr_model.predict(X_test_tfidf)
lr_acc = accuracy_score(y_test, lr_pred)

# 2. Multinomial Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_pred = nb_model.predict(X_test_tfidf)
nb_acc = accuracy_score(y_test, nb_pred)

print(f"Logistic Regression Accuracy: {lr_acc * 100:.2f}%")
print(f"Multinomial Naive Bayes Accuracy: {nb_acc * 100:.2f}%")

## 5. Evaluation & Confusion Matrix

In [ ]:
print("Classification Report (Logistic Regression):")
print(classification_report(y_test, lr_pred, target_names=['Negative', 'Positive']))

# Confusion Matrix Plot
cm = confusion_matrix(y_test, lr_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'], 
            yticklabels=['Negative', 'Positive'])
plt.title("Test Confusion Matrix")
plt.ylabel("Actual Label")
plt.xlabel("Predicted Label")
plt.show()

## 6. Live Inference & Custom Predictions

In [ ]:
def analyze_review(review_text: str):
    cleaned = clean_text(review_text)
    vec = vectorizer.transform([cleaned])
    pred = lr_model.predict(vec)[0]
    prob = lr_model.predict_proba(vec)[0]
    
    sentiment = "Positive 😊" if pred == 1 else "Negative 😞"
    confidence = prob[1] if pred == 1 else prob[0]
    
    print(f"Review:     \"{review_text}\"")
    print(f"Sentiment:  {sentiment}")
    print(f"Confidence: {confidence * 100:.2f}%")
    print("-" * 50)

# Test sample cases
analyze_review("This film was a magnificent work of art with breathtaking scenes.")
analyze_review("The script was terribly written and the acting was atrocious.")
analyze_review("An enjoyable movie with great humor and fun characters.")